In [1]:
import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse.csgraph import shortest_path

from simulator import MetapopulationSIRSolver
from cut_graph import EasyCutGraph

from itertools import product
from time import time
from pdb import set_trace
import os.path as osp

In [2]:
# Network parameters
mob_column = "Max. Number of Routes"  # Mobility column in edgelist
pop_column = "Population"  # Populaiton column in nodelist
unit_mob = 200  # Number of passangeres for each routes (people/days/routes)

In [3]:
basic_rep = 2.0
r_time = 14.0
init_i_pop = 5.0  # Initial infected populaiton (people)

In [4]:
# Simulation parameters
max_time = 10 * 365.0  # Maximum simulation time (days)
tolerance = 1e-11  # Numerical solver & terminaiton tolerance

In [5]:
# Edge cutting parameters
num_cuts = 2310  # Number of edge cut numbers
cut_method = "uniform_random"  # Edge cutting method
cut_seed = 0  # Number of different edge cuts for a single edge cut number

In [ ]:
edgelist = pd.read_csv(osp.join("data", "edgelist_symmetric.csv"))
nodelist = pd.read_csv(osp.join("data", "nodelist_connected.csv"))

edgelist["mobility"] = edgelist[mob_column] * unit_mob
nodelist = nodelist.set_index("ID")

node_attr_dict = {}
for i in nodelist.index:
    node_attr_dict[i] = {"population": nodelist.loc[i, pop_column]}

orig_graph = nx.from_pandas_edgelist(
    edgelist,
    "Source",
    "Target",
    edge_attr="mobility",
    create_using=EasyCutGraph,
)
nx.set_node_attributes(orig_graph, node_attr_dict)

num_edges = orig_graph.number_of_edges()
num_nodes = orig_graph.number_of_nodes()

In [7]:
orig_graph.nodes

NodeView((6, 5, 11, 1, 2, 8, 12, 0, 16, 17, 18, 4, 20, 22, 15, 23, 7, 24, 26, 28, 10, 29, 30, 31, 32, 33, 34, 35, 9, 13, 21, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 14, 54, 55, 56, 45, 57, 27, 58, 59, 60, 61, 62, 19, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 25, 109, 110, 111, 112, 113, 114, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 115, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 79, 213, 3, 214, 215, 216, 217, 218, 219, 

In [8]:
new_graph = EasyCutGraph()

nodes_list = [(node, {"population": nodelist.loc[node, pop_column]}) for node in nodelist.index]

new_graph.add_nodes_from(nodes_list)

edges_list = [(edgelist.loc[edge, "Source"], edgelist.loc[edge, "Target"], edgelist.loc[edge, "mobility"]) for edge in edgelist.index]

new_graph.add_weighted_edges_from(edges_list, weight='mobility')

In [9]:
new_graph.nodes

NodeView((0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 

In [12]:
nx.adjacency_matrix(new_graph, weight="mobility", dtype=np.float64)[5, 6]

200.0

In [15]:
total_pops = np.array(list(nx.get_node_attributes(new_graph, "population").values()))
total_pops[5]

13728.5

In [1]:
import numpy as np

In [5]:
A = np.array([[1, 2], [3, 4]])
b = np.array([1, 2])
A / b.reshape(-1, 1)

array([[1. , 2. ],
       [1.5, 2. ]])